<a href="https://colab.research.google.com/github/Datkhoo25/insurance_risk_prediction/blob/main/Deployment_Saving_Preprocessing_pkl_pynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import RFE
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
import pickle

# Load your data
train_path = '/content/drive/MyDrive/Colab Notebooks/Risk Prediction/train.csv'
train_df_full = pd.read_csv(train_path, index_col='Id')

# Split the data into features and target
data_target_full = train_df_full.iloc[:, -1]
data_df_full = train_df_full.iloc[:, :-1]

# Split the data into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(data_df_full, data_target_full, test_size=0.2, random_state=42)

# Identify categorical and numerical columns
categorical_cols = X_train.select_dtypes(include=['int64']).columns.tolist()
numerical_cols = X_train.select_dtypes(include=['float64']).columns.tolist()

# Identify already one-hot encoded columns
already_one_hot_encoded_cols = [col for col in categorical_cols if X_train[col].nunique() == 2 and set(X_train[col].unique()) == {0, 1}]

# Remove already one-hot encoded columns from categorical columns list
categorical_cols = [col for col in categorical_cols if col not in already_one_hot_encoded_cols]

# Define the preprocessing pipeline
preprocessing_pipeline = ColumnTransformer(transformers=[
    ('num', Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('passthrough', 'passthrough')
    ]), numerical_cols),
    ('cat', Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore'))
    ]), categorical_cols),
    ('already_one_hot', 'passthrough', already_one_hot_encoded_cols)
])

# Define the full pipeline with feature selection
full_pipeline = Pipeline(steps=[
    ('preprocessing', preprocessing_pipeline),
    ('feature_selection', RFE(estimator=DecisionTreeClassifier(), n_features_to_select=200, step=100))
])

# Fit the full pipeline on the training data
full_pipeline.fit(X_train, y_train)

# Transform the training and validation data using the fitted pipeline
X_train_transformed = full_pipeline.transform(X_train)
X_val_transformed = full_pipeline.transform(X_val)

# Save the full pipeline using Pickle
with open('/content/drive/MyDrive/Colab Notebooks/Risk Prediction/full_pipeline.pkl', 'wb') as file:
    pickle.dump(full_pipeline, file)

# Save the transformed training and validation data along with their target variables in a dictionary
data_dict = {
    'X_train_transformed': X_train_transformed,
    'X_val_transformed': X_val_transformed,
    'y_train': y_train,
    'y_val': y_val
}

with open('/content/drive/MyDrive/Colab Notebooks/Risk Prediction/transformed_data.pkl', 'wb') as file:
    pickle.dump(data_dict, file)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
